# 05.18 - ML Synthesis & Review

**Phase:** 05 - Machine Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

This is the **capstone** for Phase 05. We bring together everything: problem formulation, data prep, model selection, evaluation, tuning, and interpretation - in one end-to-end ML project.

## 2. Why Does This Matter?

This mini-project demonstrates the full ML workflow and lets you apply everything you've learned independently.

## 3. Prerequisites

- All of Phase 05 (Units 05.1-05.17)

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Build an end-to-end ML pipeline
- Compare multiple models
- Tune hyperparameters
- Interpret results

## 5. Mental Model

Full workflow:

1. Formulate the problem
2. Explore and prepare data
3. Split (no leakage)
4. Train multiple models
5. Evaluate with proper metrics
6. Tune the best model
7. Interpret and report


## 6. The Dataset

Use a real-ish classification dataset: predict diabetes.


In [ ]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

np.random.seed(42)
# Use a synthetic classification dataset for a clean demo
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=1000, n_features=15, n_informative=8, n_redundant=3, random_state=42)
print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Class balance: {y.mean():.2f}")


## 7. Split

Split into train and test, stratified.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Train balance: {y_train.mean():.2f}, Test balance: {y_test.mean():.2f}")


## 8. Compare Multiple Models

Train several models and compare with cross-validation.


In [ ]:
models = {
    'Logistic': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'SVM': make_pipeline(StandardScaler(), SVC(probability=True, random_state=42)),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    cv = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')
    results[name] = cv.mean()
    print(f"{name:18s}: CV ROC-AUC={cv.mean():.3f} +/- {cv.std():.3f}")

best_name = max(results, key=results.get)
print(f"\nBest model: {best_name}")


## 9. Tune the Best Model

Use grid search to tune the best model.


In [ ]:
if best_name == 'Gradient Boosting':
    param_grid = {'n_estimators': [50, 100], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 5]}
    base = GradientBoostingClassifier(random_state=42)
else:
    param_grid = {'n_estimators': [50, 100], 'max_depth': [5, 10]}
    base = RandomForestClassifier(random_state=42)

grid = GridSearchCV(base, param_grid, cv=3, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)
print(f"Best params: {grid.best_params_}")
print(f"Best CV ROC-AUC: {grid.best_score_:.3f}")


## 10. Final Evaluation on Test

Evaluate the tuned model once on the held-out test set.


In [ ]:
y_pred = grid.predict(X_test)
y_proba = grid.predict_proba(X_test)[:, 1]

print(f"Test accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Test precision: {precision_score(y_test, y_pred):.3f}")
print(f"Test recall:    {recall_score(y_test, y_pred):.3f}")
print(f"Test F1:        {f1_score(y_test, y_pred):.3f}")
print(f"Test ROC-AUC:   {roc_auc_score(y_test, y_proba):.3f}")
print("\nThe test set was used only once, at the end.")


## 11. Interpret the Model

Feature importance for the best model.


In [ ]:
if hasattr(grid.best_estimator_, 'feature_importances_'):
    importances = grid.best_estimator_.feature_importances_
    top = np.argsort(importances)[::-1][:5]
    print("Top 5 features by importance:")
    for i in top:
        print(f"  Feature {i}: {importances[i]:.3f}")
else:
    print("Model doesn't expose feature importance directly.")


## 12. Failure Case: What Could Go Wrong?

Common pitfalls in an end-to-end ML project.


In [ ]:
print("Common pitfalls:")
print("  1. Data leakage (preprocessing before split).")
print("  2. Tuning on the test set.")
print("  3. Using accuracy on imbalanced data.")
print("  4. Not scaling for distance-based models.")
print("  5. Overfitting with too many features.")
print("\nWe avoided all of these in this project.")


## 13. Debugging: Common Errors

- **Leakage**: split before preprocessing.
- **Overfitting**: tune with CV, not test.
- **Wrong metric**: match to business goal.

## 14. Real-World Considerations

- Use pipelines to prevent leakage.
- Validate on a separate set, test once.
- Document model choices and results.

## 15. Common Mistakes

- Tuning on test.
- Not comparing to a baseline.
- Ignoring class imbalance.

## 16. When NOT to Use

- When a simple baseline suffices.
- When interpretability outweighs accuracy.

## 17. Challenge

Add a stacking ensemble to the comparison and see if it beats the best single model.


In [ ]:
# Challenge: stacking ensemble
from sklearn.ensemble import StackingClassifier

stack = StackingClassifier(
    estimators=[
        ('lr', make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))),
        ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
        ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
    ],
    final_estimator=LogisticRegression(max_iter=1000)
)
stack.fit(X_train, y_train)
stack_auc = roc_auc_score(y_test, stack.predict_proba(X_test)[:, 1])
print(f"Stacking test ROC-AUC: {stack_auc:.3f}")
print(f"Best single model:     {roc_auc_score(y_test, grid.predict_proba(X_test)[:, 1]):.3f}")
print("\nStacking can beat the best single model.")


## 18. Closed-Book Recall

Without looking back:

1. List the steps of an end-to-end ML project.
2. Why split before preprocessing?
3. How do you choose the best model?
4. Why test only once?

## 19. Teach-Back Questions

Explain to another person:

- The full ML workflow.
- How to avoid leakage and overfitting.

## 20. Summary

You completed an end-to-end ML project: formulate, prepare, split, compare models, tune, evaluate, and interpret. You can now apply ML independently.

## 21. Further Experiment

- Try a real dataset.
- Add feature engineering.
- Deploy the model.

## 22. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, pandas, scikit-learn
OUTPUTS: PASS
LAST VERIFIED: 2026-08-28
```
